# 🤖 OCP Bionic Judge — Model Training & Comparison
> Ce notebook entraîne et compare 3 modèles de détection d'anomalies non-supervisés :
**Isolation Forest**, **One-Class SVM**, et **Local Outlier Factor**.

> ⚠️ **Notebook exploratoire (prototype).** Exploration rapide à 15 features comparant IF / OC-SVM / LOF. La **pipeline de production** (`src/features/feature_engineering.py`) utilise 24 features ciblées et compare IF / OC-SVM / **HDBSCAN** ; l'Isolation Forest est déployé (AUC-ROC 0.82), One-Class SVM étant le leader AUC (0.93). La source de vérité des chiffres est `reports/model_comparison.json` (généré par `src/models/train.py`) et l'ADR-003 — pas ce notebook.

In [ ]:
import sqlite3
import json
import time
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import ParameterGrid
import joblib
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('..')
DB_PATH = BASE_DIR / 'data' / 'ocp_bionic.db'
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## 1. Chargement et Préparation des Données


In [ ]:
conn = sqlite3.connect(str(DB_PATH))
readings = pd.read_sql('SELECT * FROM sensor_readings ORDER BY machine_id, timestamp', conn)
anomalies = pd.read_sql('SELECT DISTINCT timestamp, machine_id FROM anomalies', conn)
conn.close()

SENSORS = ['temperature', 'vibration', 'pression', 'courant', 'rpm']
readings = readings.dropna(subset=SENSORS)

for s in SENSORS:
    readings[f'{s}_roll_mean'] = readings.groupby('machine_id')[s].transform(lambda x: x.rolling(10, min_periods=1).mean())
    readings[f'{s}_roll_std'] = readings.groupby('machine_id')[s].transform(lambda x: x.rolling(10, min_periods=1).std().fillna(0))

FEATURE_COLS = [col for col in readings.columns if any(s in col for s in SENSORS)]
X = readings[FEATURE_COLS].fillna(0).values

anomaly_keys = set(zip(anomalies['machine_id'], anomalies['timestamp']))
y = np.array([-1 if (row['machine_id'], row['timestamp']) in anomaly_keys else 1 for _, row in readings.iterrows()])

print(f'X shape    : {X.shape}')
print(f'Anomaly rate: {(y==-1).mean():.2%}')

## 2. Grid Search — Isolation Forest


In [ ]:
mlflow.set_tracking_uri(str(BASE_DIR / 'mlruns'))
mlflow.set_experiment('ocp_bionic_notebook')

if_results = []
for params in ParameterGrid({'n_estimators':[100,200],'contamination':[0.03,0.05],'max_features':[0.8,1.0]}):
    model = IsolationForest(random_state=42, n_jobs=-1, **params)
    model.fit(X)
    preds = model.predict(X)
    f1 = f1_score((y==-1).astype(int), (preds==-1).astype(int), zero_division=0)
    if_results.append({**params, 'f1': round(f1,4)})

if_df = pd.DataFrame(if_results).sort_values('f1', ascending=False)
print('Isolation Forest Grid Search Results:')
print(if_df.to_string())

## 3. Comparaison ROC des 3 Modèles


In [ ]:
best_if_params = if_df.iloc[0][['n_estimators','contamination','max_features']].to_dict()
best_if = IsolationForest(random_state=42, n_jobs=-1, **{k: (int(v) if k=='n_estimators' else v) for k,v in best_if_params.items()})
best_if.fit(X)

X_sub = X[:20000]
y_sub = y[:20000]
best_svm = OneClassSVM(nu=0.05, kernel='rbf', gamma='scale')
best_svm.fit(X_sub)

best_lof = LocalOutlierFactor(novelty=True, n_neighbors=20, contamination=0.05, n_jobs=-1)
best_lof.fit(X)

models_eval = {'IsolationForest': (best_if, X), 'OneClassSVM': (best_svm, X_sub), 'LOF': (best_lof, X)}
y_vals = {'IsolationForest': y, 'OneClassSVM': y_sub, 'LOF': y}

fig = go.Figure()
for name, (mdl, Xm) in models_eval.items():
    ym = y_vals[name]
    scores = mdl.decision_function(Xm)
    y_bin = (ym == -1).astype(int)
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_bin, -scores)
    auc = roc_auc_score(y_bin, -scores)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'{name} (AUC={auc:.3f})', mode='lines'))

fig.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash='dash'), name='Random', showlegend=True))
fig.update_layout(title='Courbes ROC — Comparaison des 3 modèles', xaxis_title='FPR', yaxis_title='TPR', height=450)
fig.show()

## 4. Matrice de Confusion et Rapport Classification


In [ ]:
for name, (mdl, Xm) in models_eval.items():
    ym = y_vals[name]
    preds = mdl.predict(Xm)
    y_bin = (ym==-1).astype(int)
    p_bin = (preds==-1).astype(int)
    print(f'\n=== {name} ===')
    print(classification_report(y_bin, p_bin, target_names=['Normal','Anomalie']))
    cm = confusion_matrix(y_bin, p_bin)
    fig = px.imshow(cm, text_auto=True, labels=dict(x='Prédit', y='Réel'),
                   x=['Normal','Anomalie'], y=['Normal','Anomalie'],
                   title=f'Matrice de confusion — {name}', color_continuous_scale='Blues')
    fig.show()

## 5. Analyse des Hyperparamètres Optimaux


In [ ]:
fig = px.parallel_coordinates(if_df, color='f1', dimensions=['n_estimators','contamination','max_features','f1'],
                              color_continuous_scale='Viridis',
                              title='Impact des hyperparamètres — Isolation Forest')
fig.show()

## 6. Sélection du Modèle Final et Sauvegarde


In [ ]:
metrics_summary = []
for name, (mdl, Xm) in models_eval.items():
    ym = y_vals[name]
    preds = mdl.predict(Xm)
    y_bin = (ym==-1).astype(int)
    p_bin = (preds==-1).astype(int)
    scores = mdl.decision_function(Xm)
    metrics_summary.append({
        'model': name,
        'f1': round(f1_score(y_bin, p_bin, zero_division=0),4),
        'precision': round(precision_score(y_bin, p_bin, zero_division=0),4),
        'recall': round(recall_score(y_bin, p_bin, zero_division=0),4),
        'auc_roc': round(roc_auc_score(y_bin, -scores),4),
    })

metrics_df = pd.DataFrame(metrics_summary).set_index('model')
print(metrics_df)

best_model_name = metrics_df['f1'].idxmax()
print(f'\n✅ Best model: {best_model_name} (F1={metrics_df.loc[best_model_name,"f1"]})')

best_model_obj = models_eval[best_model_name][0]
joblib.dump({'model': best_model_obj, 'name': best_model_name, 'feature_cols': FEATURE_COLS}, str(MODEL_DIR / 'best_model.joblib'))
print(f'Model saved to {MODEL_DIR}/best_model.joblib')

## 7. Comparaison Visuelle des Métriques


In [ ]:
fig = px.bar(metrics_df.reset_index().melt(id_vars='model'), x='model', y='value', color='variable',
             barmode='group', title='Comparaison des métriques — 3 modèles',
             labels={'value': 'Score', 'variable': 'Métrique'})
fig.show()

## Conclusion

Sur cette exploration à 15 features, l'Isolation Forest et le LOF arrivent en tête du F1.

> ⚠️ **En production, les chiffres et le choix de modèle diffèrent** (voir `reports/model_comparison.json`). La pipeline officielle (24 features ciblées) donne : Isolation Forest AUC 0.82 (déployé, car seul compatible SHAP TreeExplainer + latence minimale), One-Class SVM AUC 0.93 (leader AUC), HDBSCAN AUC 0.75. L'Isolation Forest est déployé pour l'explicabilité et la latence (ADR-003, ADR-006), pas pour son F1.